# CUDA check of `release/` on a Colab T4

Runs the `--backend hf` path of `signal_b.py` (never run before) on 5 rows per language for
`gemma4-12b-4bit` and `qwen3-8b-4bit`, and optionally `signal_a.py --encoder xlmr` on CUDA.

Before running:
1. Runtime > Change runtime type > **T4 GPU**.
2. Upload the prepared folder `lcp_release` (only the four scripts, `README.md`, `requirements.txt`
   and `predictions/`; never `release/` itself, whose `data/` links to the datasets) to Google Drive
   as `MyDrive/lcp_release/`.
3. Hugging Face token with access to `MLSP2024/MLSP2024`: either a Colab secret named `HF_TOKEN`,
   or a one-line file `MyDrive/lcp_colab_out/hf_token.txt` (delete it afterwards).

Everything worth keeping is written to `MyDrive/lcp_colab_out/` (no dataset text is written there:
predictions are keyed by row index, lemmas are stored as hashes). `files.download()` does not work
from the VSCode Colab extension, so take the results from the Drive web interface.
Every heavy step runs in a subprocess, so the packages installed from `requirements.txt` are the
ones used, not whatever the kernel had imported.

In [1]:
# 1. Drive and paths. HF_HOME must be set before anything imports huggingface_hub.
import os, sys, time, shutil, hashlib, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
DRIVE_RELEASE = '/content/drive/MyDrive/lcp_release'
OUT_DIR = '/content/drive/MyDrive/lcp_colab_out'
REL = '/content/release'
os.makedirs(OUT_DIR, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_home'          # model weights stay on the local disk (~40 GB)
os.environ['MLSP_DIR'] = '/content/mlsp'            # MLSP is downloaded here, never to Drive
os.environ['STANZA_RESOURCES_DIR'] = '/content/stanza_resources'
os.environ['PYTHONUNBUFFERED'] = '1'
print(sorted(os.listdir(DRIVE_RELEASE)))

Mounted at /content/drive
['README.md', 'data.py', 'fuse.py', 'predictions', 'requirements.txt', 'signal_a.py', 'signal_b.py']


In [6]:
# 2. Hugging Face token (Colab secret, else a file in Drive, else prompt)
tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('no Colab secret:', type(e).__name__)
p = os.path.join(OUT_DIR, 'hf_token.txt')
if not tok and os.path.exists(p):
    tok = open(p).read().strip()
if not tok:
    from getpass import getpass
    tok = getpass('HF token: ').strip()
os.environ['HF_TOKEN'] = tok
print('token set:', bool(tok))

no Colab secret: TimeoutException
token set: True


In [3]:
# 3. Fresh copy of release/ (no data/, no cache/); the signal B cache lives in Drive
if os.path.exists(REL):
    shutil.rmtree(REL)
shutil.copytree(DRIVE_RELEASE, REL, ignore=shutil.ignore_patterns('data', 'cache', '__pycache__', '.DS_Store'))
os.makedirs(os.path.join(OUT_DIR, 'cache'), exist_ok=True)
os.symlink(os.path.join(OUT_DIR, 'cache'), os.path.join(REL, 'cache'))
for f in ('data.py', 'signal_a.py', 'signal_b.py', 'fuse.py', 'requirements.txt'):
    print(f, hashlib.sha256(open(os.path.join(REL, f), 'rb').read()).hexdigest()[:16])

data.py 0a34566a156b00e1
signal_a.py beee0f089b1287c1
signal_b.py b431d8bd01b74ce1
fuse.py bd883074499656a1
requirements.txt c4ec90b690c9d342


In [5]:
# 4. Install exactly requirements.txt (several minutes: torch is reinstalled)
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REL,
                   capture_output=True, text=True)
print('pip exit', r.returncode, f'{time.time()-t0:.0f}s'); print(r.stdout[-2000:]); print(r.stderr[-3000:])
info = subprocess.run([sys.executable, '-c', '''
import sys, json, torch, transformers, bitsandbytes, accelerate, numpy, pandas, sklearn
print(json.dumps({"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
  "gpu": torch.cuda.get_device_name(0), "capability": torch.cuda.get_device_capability(0),
  "bf16_supported": torch.cuda.is_bf16_supported(), "transformers": transformers.__version__,
  "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__,
  "numpy": numpy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__}))
'''], capture_output=True, text=True)
print(info.stdout, info.stderr[-2000:])
open(os.path.join(OUT_DIR, 'env_info.json'), 'w').write(info.stdout)
subprocess.run(['nvidia-smi'])

pip exit 0 153s
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 33.9 MB/s eta 0:00:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.3 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 whi

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [12]:
# 4b. Colab's torchvision/torchaudio are built for its own torch; once requirements.txt replaces
# torch they break `import transformers` ("operator torchvision::nms does not exist").
# Neither is used here.
!{sys.executable} -m pip uninstall -y torchvision torchaudio


Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [13]:
# 5. Helper that runs signal_b.run() in a subprocess and also logs, per reading, the raw probability
#    mass on the digits and the wall time (neither is stored by signal_b.py itself).
RUN_B = r'''
import sys, time, csv, os
sys.path.insert(0, "/content/release")
import torch, data, signal_b
key, out_dir = sys.argv[1], sys.argv[2]
log = []
orig = signal_b.Rater.digit_probs
def logged(self, content):
    t = time.time()
    p, raw = orig(self, content)
    log.append((raw, time.time() - t))
    return p, raw
signal_b.Rater.digit_probs = logged
t0 = time.time()
signal_b.run(key, list(signal_b.PROMPTS), False, "hf", None, 5)
part = os.path.join(data.CACHE_DIR, f"signal_b_{key}_hf.part.csv")
rows = list(csv.reader(open(part)))
hdr, body = rows[0], rows[1:]
new = body[len(body) - len(log):]
with open(os.path.join(out_dir, f"raw_{key}_hf.csv"), "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["prompt", "idx", "raw", "seconds"])
    for r, (raw, s) in zip(new, log):
        w.writerow([r[1], r[3], repr(raw), f"{s:.4f}"])
print(f"{key}: {len(log)} readings in {time.time()-t0:.0f}s total, "
      f"peak GPU memory {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
'''
open('/content/run_b.py', 'w').write(RUN_B)

def run_b(key):
    t0 = time.time()
    r = subprocess.run([sys.executable, '/content/run_b.py', key, OUT_DIR], cwd=REL, capture_output=True, text=True)
    print('exit', r.returncode, f'{time.time()-t0:.0f}s')
    print(r.stdout[-4000:]); print(r.stderr[-6000:])
    # the documented command: everything is cached now, so this checks the CLI, the default
    # backend on a non-Apple machine, and the resume path
    r = subprocess.run([sys.executable, 'signal_b.py', '--model', key, '--limit', '5'], cwd=REL,
                       capture_output=True, text=True)
    print('CLI exit', r.returncode); print(r.stdout[-1500:]); print(r.stderr[-1500:])

In [14]:
# 6. Gemma 4 12B (~24 GB download, then 150 readings)
run_b('gemma4-12b-4bit')

exit 0 815s
[B] gemma4-12b-4bit: 150 readings to do (0 cached), backend hf
[B] 100/150  2.05s per reading
[B] 150/150  2.04s per reading
[B] partial run, kept in /content/release/cache/signal_b_gemma4-12b-4bit_hf.part.csv; predictions/ is updated only by a complete run
gemma4-12b-4bit: 150 readings in 805s total, peak GPU memory 7.53 GiB

  | 368/677 [00:57<00:58,  5.29it/s]
Loading weights: 100%|██████████| 677/677 [01:36<00:00,  7.01it/s]

CLI exit 0
[B] gemma4-12b-4bit: 0 readings to do (150 cached), backend hf
[B] partial run, kept in /content/release/cache/signal_b_gemma4-12b-4bit_hf.part.csv; predictions/ is updated only by a complete run




In [15]:
# 7. Qwen3 8B (~16 GB download, then 150 readings)
run_b('qwen3-8b-4bit')

exit 0 553s
[B] qwen3-8b-4bit: 150 readings to do (0 cached), backend hf
[B] 100/150  1.43s per reading
[B] 150/150  1.46s per reading
[B] partial run, kept in /content/release/cache/signal_b_qwen3-8b-4bit_hf.part.csv; predictions/ is updated only by a complete run
qwen3-8b-4bit: 150 readings in 544s total, peak GPU memory 5.93 GiB

  | 155/399 [00:31<00:29,  8.40it/s]
Loading weights: 100%|██████████| 399/399 [01:06<00:00,  6.00it/s]

CLI exit 0
[B] qwen3-8b-4bit: 0 readings to do (150 cached), backend hf
[B] partial run, kept in /content/release/cache/signal_b_qwen3-8b-4bit_hf.part.csv; predictions/ is updated only by a complete run




## Optional: signal A on CUDA (XLM-R base, ten languages)

Writes `signal_a_cuda.csv`, the embeddings, and a hash of every lemma to Drive, so the run can be
compared with the Apple-silicon one without copying any dataset text. Roughly 10-15 minutes.

In [16]:
# 8. signal_a.py --encoder xlmr on CUDA; lemmas and embeddings are cached on the local disk
RUN_A = r'''
import sys, os, time, hashlib, shutil
sys.path.insert(0, "/content/release")
import numpy as np, pandas as pd
import data, signal_a
out_dir = sys.argv[1]
data.CACHE_DIR = "/content/cache_a"
signal_a.OUT = os.path.join(out_dir, "signal_a_cuda.csv")
t0 = time.time()
signal_a.run("xlmr", False)
print(f"signal A total {time.time()-t0:.0f}s")
lem = pd.read_csv(os.path.join(data.CACHE_DIR, "lemmas.csv"), keep_default_na=False, dtype=str)
h = lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]
pd.DataFrame({"key": [h("\x1f".join(r)) for r in lem[["part", "lang", "sent", "token"]].values],
              "lemma": [h(x) for x in lem["lemma"]]}).to_csv(os.path.join(out_dir, "lemma_hashes_cuda.csv"), index=False)
for f in os.listdir(data.CACHE_DIR):
    if f.startswith("emb_") and f.endswith(".npy"):
        shutil.copy(os.path.join(data.CACHE_DIR, f), os.path.join(out_dir, "cuda_" + f))
'''
open('/content/run_a.py', 'w').write(RUN_A)
t0 = time.time()
r = subprocess.run([sys.executable, '/content/run_a.py', OUT_DIR], cwd=REL, capture_output=True, text=True)
print('exit', r.returncode, f'{time.time()-t0:.0f}s'); print(r.stdout[-5000:]); print(r.stderr[-5000:])

exit 0 330s
[A] xlmr: embedded 8423/8423 complex_en rows in 85s
[A] xlmr: embedded 5624/5624 eval rows in 56s
[A] encoder xlmr on cuda | CompLex 8423/8423 | eval 5624/5624
  ca   n= 445  r(p_en)=+0.4133  r(p_multi)=+0.4348
  de   n= 570  r(p_en)=+0.6641  r(p_multi)=+0.6503
  en   n= 570  r(p_en)=+0.7434  r(p_multi)=+0.7434
  es   n= 593  r(p_en)=+0.6521  r(p_multi)=+0.6654
  fil  n= 570  r(p_en)=+0.4162  r(p_multi)=+0.4115
  fr   n= 569  r(p_en)=+0.5283  r(p_multi)=+0.5368
  it   n= 570  r(p_en)=+0.6249  r(p_multi)=+0.6291
  ja   n= 570  r(p_en)=+0.5993  r(p_multi)=+0.6003
  pt   n= 567  r(p_en)=+0.7663  r(p_multi)=+0.7638
  si   n= 600  r(p_en)=+0.2818  r(p_multi)=+0.2600
-> /content/drive/MyDrive/lcp_colab_out/signal_a_cuda.csv
signal A total 327s


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18618.90it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias

In [17]:
# 9. What to bring back: everything in MyDrive/lcp_colab_out except hf_token.txt
for root, _, files in os.walk(OUT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p)/1e6:9.2f} MB  {os.path.relpath(p, OUT_DIR)}')

    25.88 MB  cuda_emb_xlmr_complex_en_8423.npy
     0.01 MB  cuda_emb_xlmr_complex_en_8423_ok.npy
    17.28 MB  cuda_emb_xlmr_eval_5624.npy
     0.01 MB  cuda_emb_xlmr_eval_5624_ok.npy
     0.00 MB  env_info.json
     0.48 MB  lemma_hashes_cuda.csv
     0.01 MB  raw_gemma4-12b-4bit_hf.csv
     0.01 MB  raw_qwen3-8b-4bit_hf.csv
     0.32 MB  signal_a_cuda.csv
     0.02 MB  cache/signal_b_gemma4-12b-4bit_hf.part.csv
     0.02 MB  cache/signal_b_qwen3-8b-4bit_hf.part.csv
